## Loading the Data

In [204]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn xgboost streamlit torch skl2onnx

Python(98650) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Using cached skl2onnx-1.20.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached onnx-1.21.0-cp312-abi3-macosx_12_0_universal2.whl.metadata (8.5 kB)
Using cached skl2onnx-1.20.0-py3-none-any.whl (317 kB)
Using cached onnx-1.21.0-cp312-abi3-macosx_12_0_universal2.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 3.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3.13 install --upgrade pip


In [205]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [206]:
# Read the data, lines set to true so that it is understood that each line is own JSON object

portfolio = pd.read_json('data/portfolio.json', lines = True)
profile = pd.read_json('data/profile.json', lines=True)
transcript = pd.read_json('data/transcript.json', lines=True)

With our datasets loaded in, let's now look at the structure of data and see if we can extract anything interesting.

In [207]:
print(portfolio.shape)
portfolio.info()
portfolio.describe()

(10, 6)
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reward      10 non-null     int64 
 1   channels    10 non-null     object
 2   difficulty  10 non-null     int64 
 3   duration    10 non-null     int64 
 4   offer_type  10 non-null     str   
 5   id          10 non-null     str   
dtypes: int64(3), object(1), str(2)
memory usage: 1006.0+ bytes


,reward,difficulty,duration
count,10.000000,10.000000,10.000000
mean,4.200000,7.700000,6.500000
std,3.583915,5.831905,2.321398
min,0.000000,0.000000,3.000000
25%,2.000000,5.000000,5.000000
50%,4.000000,8.500000,7.000000
75%,5.000000,10.000000,7.000000
max,10.000000,20.000000,10.000000


In [208]:
print(profile.shape)
profile.info()
profile.describe()

(17000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               17000 non-null  int64  
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 1.2 MB


,age,became_member_on,income
count,17000.000000,1.700000e+04,14825.000000
mean,62.531412,2.016703e+07,65404.991568
std,26.738580,1.167750e+04,21598.299410
min,18.000000,2.013073e+07,30000.000000
25%,45.000000,2.016053e+07,49000.000000
50%,58.000000,2.017080e+07,64000.000000
75%,73.000000,2.017123e+07,80000.000000
max,118.000000,2.018073e+07,120000.000000


We can see that the max age is 118, and that the income count is only 14825.

In [209]:
print(transcript.shape)
transcript.info()
transcript.describe()

(306534, 4)
<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   person  306534 non-null  str   
 1   event   306534 non-null  str   
 2   value   306534 non-null  object
 3   time    306534 non-null  int64 
dtypes: int64(1), object(1), str(2)
memory usage: 22.3+ MB


,time
count,306534.000000
mean,366.382940
std,200.326314
min,0.000000
25%,186.000000
50%,408.000000
75%,528.000000
max,714.000000


## Data Analysis


In [210]:
# Let's look at the value distributions for some cateogrical columns:

print(profile['gender'].value_counts())
print()
print(transcript['event'].value_counts())
print()
print(portfolio['offer_type'].value_counts())

gender
M    8484
F    6129
O     212
Name: count, dtype: int64

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

offer_type
bogo             4
discount         4
informational    2
Name: count, dtype: int64


The gender breakdown is M: 8484, F: 6129, O: 212, with 2175 null values (those are the age = 118 rows, the same customers with missing demographics).

The transcript has 4 event types. 76,277 offers were received, but only 33,579 were completed: roughly a 44% raw completion rate. This will be our target variable. Note that 138,953 transactions exist separately: customers spending money independent of any offer.

The portfolio has 4 BOGO, 4 discount, and 2 informational offers. Informational offers have no reward and can't technically be "completed". We'll need to handle those separately.

In [211]:
# Let's peek at the value column in the transcript, where offer IDs are buried.
transcript['value'].sample(20)

35664                                     {'amount': 12.38}
108922     {'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'}
63414      {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
202985     {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
271136    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
117133     {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
109810                                     {'amount': 0.85}
52648                                      {'amount': 6.83}
287427                                     {'amount': 5.38}
135047     {'offer id': '2906b810c7d4411798c6938adc9daaa5'}
234634                                     {'amount': 5.53}
174000     {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
220839     {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
261893     {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
166261                                      {'amount': 3.8}
148571     {'offer id': '5a8bc65990b245e5a138643cd4eb9837'}
267888                                  

We see that there are a few different data types:

{'offer id': '...'} : note the space in the key (most offer rows)

{'offer_id': '...'} : note the underscore (row 186219. This is a completed offer event)

{'amount': 22.79} — transaction rows

Let's see if we can detect a pattern on when the underscore occurs:

In [212]:
transcript[transcript['event'] == 'offer completed']['value'].head(15)

12658    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12672    {'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...
12679    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12692    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
12697    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12717    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12721    {'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...
12744    {'offer_id': 'f19421c1d4aa40978ebb69ca19b0e20d...
12764    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12767    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12780    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12784    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12786    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12798    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
12817    {'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...
Name: value, dtype: object

We observe that every single offer completed row uses offer_id (underscore), while offer received and offer viewed rows use offer id (space). When we later extract offer IDs, we will need to handle such cases using this information.

A few other quirks to note:

- The age max of 118 is a confirmed missing marker

- became_member_on shows as 2.016703e+07 — that's just scientific notation for 20170212. It's being treated as a number, instead of a date. We'll convert it to an actual date in cleaning.

- income ranges from 30k to 120k, mean ~65k, which is reasonable, with no obvious outliers there.


## Data Cleaning

The current issues to fix:

- Remove rows where age == 118 (missing demographic marker)
- Convert became_member_on from integer to datetime
- Standardize value column keys (offer id → offer_id)


In [213]:
# We will first confirm that the a row with age 118 indicates a dead row:

profile[profile['age'] == 118][['age', 'gender', 'income']].head(10)

,age,gender,income
0,118,NaN,NaN
2,118,NaN,NaN
4,118,NaN,NaN
6,118,NaN,NaN
7,118,NaN,NaN
9,118,NaN,NaN
10,118,NaN,NaN
11,118,NaN,NaN
17,118,NaN,NaN
23,118,NaN,NaN


In [214]:
# Confirmed. 

profile = profile[profile['age'] != 118]
print(profile.shape)

# Now profile has the valid number of customers with valid demographics.

(14825, 5)


In [215]:
# Let's now convert became_member_on to a datetime.

profile['became_member_on'] = pd.to_datetime(profile['became_member_on'], format = '%Y%m%d')
profile['became_member_on'].head()

1    2017-07-15
3    2017-05-09
5    2018-04-26
8    2018-02-09
12   2017-11-11
Name: became_member_on, dtype: datetime64[us]

In [216]:
# Let's now create a function to standardize the value column keys. 
# Below will replace offer id with offer_id

def standardize_value(d):
    if not isinstance(d, dict):
        return d

    if 'offer id' in d:
        d['offer_id'] = d.pop('offer id') # pop removes key and returns value
    return d


# Let's now apply the function across each row in transcript:

transcript['value'] = transcript['value'].apply(standardize_value)
transcript['value'].sample(15)

284933                                    {'amount': 15.23}
127697     {'offer_id': 'f19421c1d4aa40978ebb69ca19b0e20d'}
18381      {'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2'}
73077                                     {'amount': 17.96}
101427                                    {'amount': 23.53}
260671    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
161442     {'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4'}
216453     {'offer_id': '2906b810c7d4411798c6938adc9daaa5'}
120980     {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
25065     {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
107076                                    {'amount': 15.09}
85186     {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
46077                                      {'amount': 1.99}
100601                                    {'amount': 18.22}
45597                                     {'amount': 11.15}
Name: value, dtype: object

## Feature Engineering

We want to combine these tables into one, flat table where each row is one (customer, offer) pair, with features that we can feed into a model. Right now the data is split acrosss three separate tables, so we need to join them:

To build this, we will need to: 

- Extract offer events from transcript: pull out each offer received, offer viewed, offer completed row with its offer_id

- Determine if an offer was completed: for each (customer, offer) pair, was there a corresponding offer completed event?

- Join with profile: attach the customer's demographics

- Join with portfolio: attach the offer's details

- Engineer member_tenure_days — how long had they been a member when the offer was sent?


In [217]:
offer_events = transcript[transcript['event'] != 'transaction'].copy()
offer_events['offer_id'] = offer_events['value'].apply(lambda d: d.get('offer_id'))
offer_events.head()

,person,event,value,time,offer_id
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer_id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer_id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0


In [218]:
# We now extracted the offer id to its own column. Next, we will build the target varaible.
# We ask the question, for each unique (customer, offer) pair, did a offer completed event exist?

completed = offer_events[offer_events['event'] == 'offer completed'][['person', 'offer_id']].drop_duplicates()
completed['completed'] = 1 # for binary labeling
completed.head()

,person,offer_id,completed
12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,2906b810c7d4411798c6938adc9daaa5,1
12672,fe97aa22dd3e48c8b143116a8403dd52,fafdcd668e3743c1bb461111dcafc2a4,1
12679,629fc02d56414d91bca360decdfa9288,9b98b8c7a33c4b65b9aebfe6a799e6d9,1
12692,676506bad68e4161b9bbaffeb039626b,ae264e3637204a6fb9bb56bc8210ddfd,1
12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,4d5c57ea9a6940dd891ad53e9dbe8da0,1


In [219]:
# We now merge this onto the offer received rows to create a labeled dataset.abs

received = offer_events[offer_events['event'] == 'offer received'][['person', 'offer_id', 'time']].copy()

df = received.merge(completed, on=['person', 'offer_id'], how='left') # Left join so that customers who never get completed get NaN

df['completed'] = df['completed'].fillna(0).astype(int) # binary classification

df.head()

,person,offer_id,time,completed
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,1
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,0
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,0
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,0


In [220]:
# We now merge the customer demographics and order details

df = df.merge(profile, left_on='person', right_on='id', how='left') # left on and right on because of different column names for each table
df = df.merge(portfolio, left_on='offer_id', right_on='id', how='left')
df.head()

,person,offer_id,time,completed,gender,age,id_x,became_member_on,income,reward,channels,difficulty,duration,offer_type,id_y
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,1,F,75.0,78afa995795e4d85b5d9ceeca43f5fef,2017-05-09,100000.0,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,0,NaN,NaN,NaN,NaT,NaN,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,0,M,68.0,e2127556f4f64592b11af22de27a7932,2018-04-26,70000.0,2,"[web, email, mobile]",10,7,discount,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,0,NaN,NaN,NaN,NaT,NaN,2,"[web, email, mobile, social]",10,10,discount,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,0,NaN,NaN,NaN,NaT,NaN,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0


In [221]:
# Notice that id_x and id_y are redudant, they're just copies of person and offer_id, so we can drop them.
# We see that some rows have NaN demographics, which we should drop as well. 

df = df.drop(columns=['id_x', 'id_y'])
df = df.dropna(subset=['age', 'income', 'gender'])
print(df.shape)
df.head()

(66501, 13)


,person,offer_id,time,completed,gender,age,became_member_on,income,reward,channels,difficulty,duration,offer_type
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,1,F,75.0,2017-05-09,100000.0,5,"[web, email, mobile]",5,7,bogo
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,0,M,68.0,2018-04-26,70000.0,2,"[web, email, mobile]",10,7,discount
5,389bc3fa690240e798340f5a15918d5c,f19421c1d4aa40978ebb69ca19b0e20d,0,1,M,65.0,2018-02-09,53000.0,5,"[web, email, mobile, social]",5,5,bogo
7,2eeac8d8feae4a8cad5a6af0499a211d,3f207df678b143eea3cee63160fa8bed,0,0,M,58.0,2017-11-11,51000.0,0,"[web, email, mobile]",0,4,informational
8,aa4862eba776480b8bb9c68455b8c2e1,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,0,F,61.0,2017-09-11,57000.0,5,"[web, email]",20,10,discount


In [222]:
# There is one last feature that we can engineer, which is member tenure. We will use a reference date
# to track the amount of time a customer has been a member since an offer was sent.

reference_date = pd.Timestamp('2018-08-01')
df['member_tenure_days'] = (reference_date - df['became_member_on']).dt.days
df[['became_member_on', 'member_tenure_days']].head()

,became_member_on,member_tenure_days
0,2017-05-09,449
2,2018-04-26,97
5,2018-02-09,173
7,2017-11-11,263
8,2017-09-11,324


In [223]:
# Because our model can only take numbers, we need to prepare it as such:

df = df.drop(columns=['became_member_on', 'person', 'offer_id', 'time']) # drop columns we don't need

# convert channels into four binary columns:

for channel in ['email', 'web', 'mobile', 'social']:
    df[f'channel_{channel}'] = df['channels'].apply(lambda x: int(channel in x))
df = df.drop(columns=['channels'])

# Convert gender string to a number: 

df['gender'] = df['gender'].map({'M': 0, 'F': 1, 'O': 2})

# convert offer_type string to a number:

df['offer_type'] = df['offer_type'].map({'bogo': 0, 'discount': 1, 'informational': 2})

df.dtypes

completed               int64
gender                  int64
age                   float64
income                float64
reward                  int64
difficulty              int64
duration                int64
offer_type              int64
member_tenure_days      int64
channel_email           int64
channel_web             int64
channel_mobile          int64
channel_social          int64
dtype: object

## Modelling

We'll start with logistic regression as a baseline, then move to random

In [224]:
from sklearn.model_selection import train_test_split

# Let's first split our data into training and testing.

X = df.drop(columns=['completed'])
y = df['completed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(53200, 12) (13301, 12)


In [225]:
# Now for our first model:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.67      0.72      6363
           1       0.73      0.83      0.78      6938

    accuracy                           0.75     13301
   macro avg       0.76      0.75      0.75     13301
weighted avg       0.76      0.75      0.75     13301



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [226]:
# The model didn't finish optimizing, and that's because our features how different scales. Let's standardize:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # fit learns scale from training data
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.68      0.73      6363
           1       0.74      0.84      0.79      6938

    accuracy                           0.77     13301
   macro avg       0.77      0.76      0.76     13301
weighted avg       0.77      0.77      0.76     13301



In [227]:
# Let's try a random forest model. We won't need to scale as the deecision trees are split on thresholds, and not distances.
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.84      0.78      0.81      6363
           1       0.81      0.86      0.83      6938

    accuracy                           0.82     13301
   macro avg       0.82      0.82      0.82     13301
weighted avg       0.82      0.82      0.82     13301



In [228]:
# Random forest worked better. We should also try a GradientBoostingClassifier

from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbc.fit(X_train, y_train)

y_pred_gbc = gbc.predict(X_test)
print(classification_report(y_test, y_pred_gbc))

              precision    recall  f1-score   support

           0       0.82      0.73      0.77      6363
           1       0.77      0.85      0.81      6938

    accuracy                           0.79     13301
   macro avg       0.80      0.79      0.79     13301
weighted avg       0.79      0.79      0.79     13301



| Model | Accuracy | F1 (completed) |
|---|---|---|
| Logistic Regression | 77% | 0.79 |
| Gradient Boosting | 79% | 0.81 |
| Random Forest | **82%** | **0.83** |

We see that random forest scores the greatest, so we will use it for our model.

In [229]:
# Retrain with fewer trees to keep file size under GitHub's 100MB limit
import pickle
from sklearn.ensemble import RandomForestClassifier

rf_small = RandomForestClassifier(n_estimators=20, random_state=42)
rf_small.fit(X_train, y_train)

with open('model.pkl', 'wb') as f:
    pickle.dump(rf_small, f)

print("Model saved.")

Model saved.


Our model is deployed and fully useable in the app, but let's say we want to expand it to track sequences of events over time. To do this, we will use LSTM (long-term short memory) wity Pytorch.

In [230]:
import torch
import torch.nn as nn
print(torch.__version__)

2.11.0


In [231]:
# Let's reload our data into sequences
# For each offer a customer received, we want to grab their last N events
# before that offer as context.

# First we reload original transcript to get full event history:

transcript_raw = pd.read_json('data/transcript.json', lines=True)  # load original unmodified transcript

transcript_raw['offer_id'] = transcript_raw['value'].apply(
    lambda x: x.get('offer id') or x.get('offer_id') if isinstance(x, dict) else None  # extract offer id, handling both key formats
)

transcript_raw['amount'] = transcript_raw['value'].apply(
    lambda x: x.get('amount') if isinstance(x, dict) else None  # extract transaction amount, None for non-transaction rows
)

transcript_raw.head()

,person,event,value,time,offer_id,amount
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN


In [232]:
# Let's now build our sequences.abs

SEQ_LEN = 10  # number of past events to look at per customer

def build_sequence(person, offer_time):
    history = transcript_raw[
        (transcript_raw['person'] == person) &  # only this customer's events
        (transcript_raw['time'] < offer_time)    # only events before the offer
    ].tail(SEQ_LEN)  # take the last SEQ_LEN events
    
    events = []
    for _, row in history.iterrows():
        event_type = {'offer received': 0, 'offer viewed': 1, 
                      'offer completed': 2, 'transaction': 3}.get(row['event'], 0)  # encode event as number
        amount = 0.0 if pd.isna(row['amount']) else float(row['amount'])
        events.append([event_type, amount])  # each timestep is [event_type, amount]
    
    # pad with zeros if fewer than SEQ_LEN events exist
    while len(events) < SEQ_LEN:
        events.insert(0, [0, 0.0])
    
    return events

print("Function defined.")

# rebuild df with person and time kept for sequence building
df_seq = received.copy()
df_seq = df_seq.merge(completed, on=['person', 'offer_id'], how='left')  # add completed labels
df_seq['completed'] = df_seq['completed'].fillna(0).astype(int)

df_seq = df_seq.merge(profile[['id', 'gender', 'age', 'income', 'became_member_on']], 
                      left_on='person', right_on='id', how='inner')
df_seq = df_seq.dropna(subset=['age', 'income', 'gender'])
df_seq['member_tenure_days'] = (pd.Timestamp('2018-08-01') - 
                                pd.to_datetime(df_seq['became_member_on'], format='%Y%m%d')).dt.days

print(df_seq.shape)
df_seq[['person', 'time', 'completed']].head()


Function defined.
(66501, 10)


,person,time,completed
0,78afa995795e4d85b5d9ceeca43f5fef,0,1
1,e2127556f4f64592b11af22de27a7932,0,0
2,389bc3fa690240e798340f5a15918d5c,0,1
3,2eeac8d8feae4a8cad5a6af0499a211d,0,0
4,aa4862eba776480b8bb9c68455b8c2e1,0,0


In [233]:
# Now we build sequences.

print("Building sequences... this will take a few minutes.")

sequences = []
for _, row in df_seq.iterrows():  # loop over every (customer, offer) pair
    seq = build_sequence(row['person'], row['time'])  # get last 10 events before this offer
    sequences.append(seq)

print(f"Done. Built {len(sequences)} sequences.")


Building sequences... this will take a few minutes.
Done. Built 66501 sequences.


In [234]:
# We now convert these sequences to PyTorch tensors and build the LSTM:

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


X_seq = torch.tensor(sequences, dtype=torch.float32)          # shape: (66501, 10, 2)
y_seq = torch.tensor(df_seq['completed'].values, dtype=torch.float32)  # shape: (66501,)

print(X_seq.shape, y_seq.shape)

torch.Size([66501, 10, 2]) torch.Size([66501])


In [235]:
# Now we train/test split

split = int(0.8 * len(X_seq))  # 80% train, 20% test
X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
y_train_seq, y_test_seq = y_seq[:split], y_seq[split:]

# wrap in DataLoader for batching
train_ds = TensorDataset(X_train_seq, y_train_seq)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

print(f"Train: {X_train_seq.shape}, Test: {X_test_seq.shape}")


Train: torch.Size([53200, 10, 2]), Test: torch.Size([13301, 10, 2])


In [236]:
# We now define our model:

class OfferLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze(-1)  # only squeeze the last dimension

model_lstm = OfferLSTM()
print(model_lstm)



OfferLSTM(
  (lstm): LSTM(2, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


The intuition is that LSTM reads each of the 10 timesteps and updates its hidden state: think of it as a running summary of what it's seen. After the 10th step, that final summary gets passed to a single output neuron that produces a completion probability between 0 and 1.

In [237]:
# Now we train the model.pkl

print(torch.isnan(X_seq).sum())


criterion = nn.BCELoss()               # binary cross entropy loss for 0/1 classification
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)  # Adam optimizer

EPOCHS = 5

for epoch in range(EPOCHS):
    model_lstm.train()
    total_loss = 0
    for X_batch, y_batch in train_dl:          # loop over batches
        optimizer.zero_grad()                   # clear old gradients
        preds = model_lstm(X_batch)             # forward pass
        loss = criterion(preds, y_batch)        # compute loss
        loss.backward()                         # backpropagation
        optimizer.step()                        # update weights
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_dl):.4f}")


tensor(0)
Epoch 1/5 - Loss: 0.6370
Epoch 2/5 - Loss: 0.6309
Epoch 3/5 - Loss: 0.6296
Epoch 4/5 - Loss: 0.6286
Epoch 5/5 - Loss: 0.6286


In [238]:
# Loss is going down each epoch. Now we evaluate on test set:

model_lstm.eval()  # switch to evaluation mode (disables dropout etc.)

with torch.no_grad():  # don't compute gradients during evaluation
    preds_seq = model_lstm(X_test_seq)
    predicted_labels = (preds_seq >= 0.5).int()  # threshold at 0.5 to get 0/1 labels

from sklearn.metrics import classification_report
print(classification_report(y_test_seq.int(), predicted_labels))

              precision    recall  f1-score   support

           0       0.74      0.57      0.64      6552
           1       0.66      0.80      0.72      6749

    accuracy                           0.69     13301
   macro avg       0.70      0.68      0.68     13301
weighted avg       0.70      0.69      0.68     13301



Hmm.. that's lower than random forest. Let's try a new model which uses demographic features. We can concatenate the customer's age, income, and gender onto the hidden state before the final prediction layer.

In [239]:
class OfferLSTMv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=64, batch_first=True)  # bigger hidden size
        self.fc = nn.Linear(64 + 3, 32)   # 64 from LSTM + 3 demographics (age, income, gender)
        self.relu = nn.ReLU()
        self.out = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x_seq, x_demo):
        _, (hidden, _) = self.lstm(x_seq)          # run sequence through LSTM
        combined = torch.cat([hidden[-1], x_demo], dim=1)  # concatenate LSTM output + demographics
        out = self.relu(self.fc(combined))          # pass through hidden layer
        return self.sigmoid(self.out(out)).squeeze(-1)

model_v2 = OfferLSTMv2()
print(model_v2)

OfferLSTMv2(
  (lstm): LSTM(2, 64, batch_first=True)
  (fc): Linear(in_features=67, out_features=32, bias=True)
  (relu): ReLU()
  (out): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [240]:
df_seq['gender'] = df_seq['gender'].map({'M': 0, 'F': 1, 'O': 2})

demo_features = torch.tensor(
    df_seq[['age', 'income', 'gender']].values, dtype=torch.float32
)

# normalize age and income so they're on a similar scale to other features
demo_features[:, 0] = demo_features[:, 0] / 100       # age: divide by 100
demo_features[:, 1] = demo_features[:, 1] / 100000    # income: divide by 100000

print(demo_features.shape)


torch.Size([66501, 3])


In [241]:
# Now we retrain:

demo_train, demo_test = demo_features[:split], demo_features[split:]

train_ds_v2 = TensorDataset(X_train_seq, demo_train, y_train_seq)
train_dl_v2 = DataLoader(train_ds_v2, batch_size=64, shuffle=True)

# Training loop 

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model_v2.parameters(), lr=0.001)

EPOCHS = 10

for epoch in range(EPOCHS):
    model_v2.train()
    total_loss = 0
    for X_batch, demo_batch, y_batch in train_dl_v2:  # now unpacking 3 items
        optimizer.zero_grad()
        preds = model_v2(X_batch, demo_batch)          # pass both sequence and demographics
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_dl_v2):.4f}")




Epoch 1/10 - Loss: 0.6337
Epoch 2/10 - Loss: 0.6255
Epoch 3/10 - Loss: 0.6231
Epoch 4/10 - Loss: 0.6217
Epoch 5/10 - Loss: 0.6208
Epoch 6/10 - Loss: 0.6204
Epoch 7/10 - Loss: 0.6200
Epoch 8/10 - Loss: 0.6194
Epoch 9/10 - Loss: 0.6191
Epoch 10/10 - Loss: 0.6187


In [242]:
# Now evaluate: 

model_v2.eval()

with torch.no_grad():
    preds_v2 = model_v2(X_test_seq, demo_test)
    predicted_labels_v2 = (preds_v2 >= 0.5).int()

print(classification_report(y_test_seq.int(), predicted_labels_v2))

              precision    recall  f1-score   support

           0       0.73      0.57      0.64      6552
           1       0.66      0.80      0.72      6749

    accuracy                           0.69     13301
   macro avg       0.69      0.68      0.68     13301
weighted avg       0.69      0.69      0.68     13301



This model did not score any better.

# Final Results

| Model | Accuracy | F1 (completed) |
|---|---|---|
| Logistic Regression | 77% | 0.79 |
| Gradient Boosting | 79% | 0.81 |
| Random Forest | **82%** | **0.83** |
| LSTM v1 (sequence only) | 69% | 0.73 |
| LSTM v2 (sequence + demographics) | 69% | 0.72 |

The LSTM underperforms because most offers were sent at time=0, leaving 
little prior event history for the model to learn from. Random Forest wins 
by leveraging richer static features.

In [243]:
# We export best model:

import skl2onnx

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, 12]))]
onnx_model = convert_sklearn(rf_small, initial_types=initial_type)

with open('model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print("Exported.")


Exported.
